In [1]:
import numpy as np
import pandas as pd


Reading the dataset

In [2]:
df = pd.read_csv('/kaggle/input/datasets/organizations/google/google-patent-phrase-similarity-dataset/train.csv')
df.head()

,anchor,target,rating,score,context
0,aralkynyl,cholesterol,1d,0.25,C09
1,aralkynyl,aralkyl,1d,0.25,C09
2,aralkynyl,heterocycle,1c,0.25,C09
3,aralkynyl,acyl,1d,0.25,C09
4,aralkynyl,heterocyclic,1c,0.25,C09


studying the dataset

In [3]:
df.describe(include= object)

,anchor,target,rating,context
count,36473,36473,36473,36473
unique,733,29340,10,106
top,component composite coating,composition,0,H01
freq,152,24,7471,2186


checking the null values

In [4]:
df.isna().sum()

anchor     0
target     0
rating     0
score      0
context    0
dtype: int64

checking the duplicate datas in df

In [5]:
df.duplicated().sum()

np.int64(0)

Creating a input feature, combining context, target and the anchor feature

In [6]:
df['input'] = 'TEXT1: ' + df['context'] + '; TEXT2: ' + df['target'] + '; ANC1: ' + df['anchor']

In [7]:
df.head()

,anchor,target,rating,score,context,input
0,aralkynyl,cholesterol,1d,0.25,C09,TEXT1: C09; TEXT2: cholesterol; ANC1: aralkynyl
1,aralkynyl,aralkyl,1d,0.25,C09,TEXT1: C09; TEXT2: aralkyl; ANC1: aralkynyl
2,aralkynyl,heterocycle,1c,0.25,C09,TEXT1: C09; TEXT2: heterocycle; ANC1: aralkynyl
3,aralkynyl,acyl,1d,0.25,C09,TEXT1: C09; TEXT2: acyl; ANC1: aralkynyl
4,aralkynyl,heterocyclic,1c,0.25,C09,TEXT1: C09; TEXT2: heterocyclic; ANC1: aralkynyl


Making a huggingface dataset from ourdataframe

In [8]:
from datasets import Dataset,DatasetDict
ds = Dataset.from_pandas(df)             
ds

Dataset({
    features: ['anchor', 'target', 'rating', 'score', 'context', 'input'],
    num_rows: 36473
})

Taking a pre-trained model


In [9]:
model_nm = 'microsoft/deberta-v3-small'

Creation of tokenizer

In [10]:
from transformers import AutoTokenizer
tokz = AutoTokenizer.from_pretrained(model_nm)  #creating a tokenizer based on the pretrained model
tokz

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

DebertaV2Tokenizer(name_or_path='microsoft/deberta-v3-small', vocab_size=128000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[CLS]', 'eos_token': '[SEP]', 'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128000: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [11]:
def tokenize_data(x):
    return tokz(x['input'])  #tokenizing the input feature of a dataframe

Tokenization

In [12]:
tok_ds = ds.map(tokenize_data,batched=True)  #applying the tokenize function to each rows but in chunk, using batched = True 

Map:   0%|          | 0/36473 [00:00<?, ? examples/s]

In [25]:
first_row = tok_ds[0]
first_input = first_row['input']
first_ids = first_row['input_ids']
print('The first input data is:', first_input)
print('The first corresponding id is:', first_ids)

The first input data is: TEXT1: C09; TEXT2: cholesterol; ANC1: aralkynyl
The first corresponding id is: [54453, 435, 294, 716, 4505, 346, 54453, 445, 294, 9888, 346, 23702, 435, 294, 266, 17226, 9593, 63791]


In [30]:
tokz.vocab['cholesterol']

84044

In [31]:
tok_ds.rename_columns({'score':'labels'})

Dataset({
    features: ['anchor', 'target', 'rating', 'labels', 'context', 'input', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 36473
})

Splitting the dataset into train_test using train-test split

In [34]:
dds = tok_ds.train_test_split(0.20,seed = 42)    #we are using 20% of the total dataset as the test dataset
dds

DatasetDict({
    train: Dataset({
        features: ['anchor', 'target', 'rating', 'score', 'context', 'input', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 29178
    })
    test: Dataset({
        features: ['anchor', 'target', 'rating', 'score', 'context', 'input', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7295
    })
})